# Buy Signal Analysis

Add more cells below as needed (one per analyze script).

**Before running:** set `DATA_FOLDER` to the folder containing your
session output CSV files. 

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

strategies = ["MACDTarget", "MACDTrail",  "MACDFlip"]
STRATEGY = strategies[1]  # 0-MACDTarget, 1-MACDTrail, 2-MACDFlip

BASE_DIR = Path(r"C:\Git\CandleStateSessionAnalysis\data")
DATA_FOLDER = BASE_DIR / STRATEGY
OUTPUT_FOLDER = DATA_FOLDER / "output" 

In [ ]:
import pandas as pd

trade_signals = pd.read_csv(OUTPUT_FOLDER / "trade_signals.csv")
trade_signals["BuyersInControl"] = (trade_signals["Control"] == "Buyers").astype(int)

positions = pd.read_csv(OUTPUT_FOLDER / "positions.csv")
positions["TimeHeld"] = pd.to_timedelta((positions["IndexClose"] - positions["IndexOpen"]) * 2, unit="m")

line_charts_summary = pd.read_csv(OUTPUT_FOLDER / "line_chart_summary.csv")

value_cols = [
    "PriorDayHigh", "PriorDayLow", "PriorDayClose", "CurrentDayOpen",
    "OpeningGap", "OpenRangeHigh", "OpenRangeLow", "OpenRangeBips",
    "LowCloseOfDay", "HighCloseOfDay", "AverageTrendLength", "CompletedLegCount"
]  # columns B–M

trade_signals = trade_signals.merge(
    line_charts_summary[["Symbol", "SessionStart"] + value_cols],
    on=["Symbol", "SessionStart"],
    how="left",
    validate="m:1",  # each trade should match exactly one session-summary row
)

# sanity check: did every trade actually find a match?
#print(trade_signals[value_cols].isna().all(axis=1).sum(), "trades with no match")


#print(trade_signals.shape, positions.shape)
#trade_signals.dtypes
#trade_signals.head(10)
#trade_signals["Signal"].unique()
#trade_signals["Signal"].value_counts()

buy_trade_signals = trade_signals[trade_signals["Signal"] == "Buy"]
#print(buy_trade_signals.head(10))
#buy_trade_signals["PositionID"].unique()

print(f"Buy signals with null PositionID: {buy_trade_signals['PositionID'].isna().sum()}")
print(f"Buy signals with PositionID: {buy_trade_signals['PositionID'].notna().sum()}")

buy_trade_signals = buy_trade_signals[buy_trade_signals["PositionID"].notna()]
print(f"Buy signals remaining: {len(buy_trade_signals)}")


In [ ]:

positions_subset = positions[["ID", "AvgPrice", "ClosePrice", "RealizedGain", "TimeHeld"]].rename(columns={"ID": "PositionID"})
buy_trade_signals = buy_trade_signals.merge(positions_subset, on="PositionID", how="left")
buy_trade_signals["IsWin"] = (buy_trade_signals["RealizedGain"] > 0).astype(int)
buy_trade_signals[["Symbol", "PositionID", "RealizedGain", "TimeHeld", "IsWin"]]

trade_signals = trade_signals.merge(positions_subset, on="PositionID", how="left")
trade_signals["Unrealized"] = (trade_signals["Close"] - trade_signals["AvgPrice"]) * trade_signals["Shares"]

import numpy as np

conditions = [
    buy_trade_signals["AvgPrice"] > buy_trade_signals["OpenRangeHigh"],
    buy_trade_signals["AvgPrice"] < buy_trade_signals["OpenRangeLow"],
]
buy_trade_signals["OpenRangePos"] = np.select(conditions, [1, -1], default=0)


In [ ]:


candle_cols = [
    'CandleHeight', 'CandleBody', 'CandleDelta', 'CandleBodyDelta',
    'CandleBodyPct', 'UpperWickPct', 'LowerWickPct', 'TrueRange',
    'ATR', 'Volatility', 'VolatilityBps', 'VolumeRatio',"BuyersInControl"
]

trend_cols = [
    'VWAP', 'VWAPDelta', 'VWAPSlope', 
    'MarkovState', 'MarkovStateTrend', 'TrendCount', 'ADX', 'MACDHistogram',
    'SlowSlope', 'SlowLongSlope',
    'FastSlope', 'FastLongSlope', 'SlowDelta', 'FastDelta', 'MovignAverageDelta'
]

range_cols = ["OpenRangePos", "OpenRangeBips", "OpeningGap", "AverageTrendLength", "CompletedLegCount"]

buy_trade_signals[candle_cols + trend_cols + range_cols + ["IsWin"]].corr()["IsWin"].sort_values(ascending=False)


In [ ]:
buy_trade_signals.boxplot(column="ATR", by="IsWin")


In [ ]:
median_atr = buy_trade_signals["ATR"].median()
buy_trade_signals["AboveMedianATR"] = buy_trade_signals["ATR"] > median_atr
buy_trade_signals.groupby("AboveMedianATR")["IsWin"].mean()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

corr = buy_trade_signals[trend_cols + ["IsWin"]].corr()
plt.figure(figsize=(14, 12))
sns.heatmap(corr, cmap="coolwarm", vmin=-1, vmax=1, annot=True, fmt=".2f", annot_kws={"size": 6})
plt.tight_layout()
plt.show()

In [ ]:
#buy_trade_signals[["SlowMA", "FastMA"]].plot(figsize=(12, 5))
#(buy_trade_signals["SlowMA"] - buy_trade_signals["FastMA"]).describe()
#buy_trade_signals[["SlowMA", "FastMA"]]

#buy_trade_signals[["SlowDelta", "FastDelta", "MovingAverageDelta", "FastSlope", "SlowSlope", "IsWin"]].corr()["IsWin"]
#buy_trade_signals[["ATR", "FastSlope"]].corr()

after_1pm_positions_red = trade_signals[
    (trade_signals["PositionID"].notna()) &
    (trade_signals["Unrealized"] < 0) &
    (trade_signals["CandleOpen"].dt.time > pd.Timestamp("13:00:00").time()) &
    (trade_signals["CandleOpen"].dt.date < pd.Timestamp("2026-07-10").date())
]

positionIDs_red_after_1pm = after_1pm_positions_red["PositionID"].unique()
outcomes = buy_trade_signals[buy_trade_signals["PositionID"].isin(positionIDs_red_after_1pm)][["PositionID", "RealizedGain", "IsWin"]].drop_duplicates()

close_info = positions[positions["ID"].isin(positionIDs_red_after_1pm)][["ID", "Closed", "RealizedGain"]].rename(columns={"ID": "PositionID"})

close_info["CloseType"] = close_info["Closed"].dt.time.apply(
    lambda t: "EOD" if t == pd.Timestamp("15:54:00").time() else "PreEOD"
)

eod_positions = close_info[close_info["CloseType"] == "EOD"]["PositionID"]

eod_trajectory = trade_signals[
    trade_signals["PositionID"].isin(eod_positions)
][["PositionID", "CandleOpen", "Close", "SlowSlope", "FastSlope", "Unrealized"]].sort_values(["PositionID", "CandleOpen"])

eod_trajectory.to_csv(OUTPUT_FOLDER / "eod_trajectory.csv", index=False)



In [ ]:
all_negative = eod_trajectory.groupby("PositionID")["FastSlope"].apply(lambda x: (x < 0).all())
print(f"Positions where FastSlope was negative every candle: {all_negative.sum()} of {len(all_negative)}")

mostly_negative = eod_trajectory.groupby("PositionID")["FastSlope"].apply(lambda x: (x < 0).mean() > 0.5)
print(f"Positions where FastSlope was negative more than half the time: {mostly_negative.sum()} of {len(mostly_negative)}")

finished_positive = eod_trajectory.sort_values("CandleOpen").groupby("PositionID")["FastSlope"].last() > 0
print(f"Positions where FastSlope finished positive: {finished_positive.sum()} of {len(finished_positive)}")

summary = pd.DataFrame({
    "AllNegative": eod_trajectory.groupby("PositionID")["FastSlope"].apply(lambda x: (x < 0).all()),
    "MostlyNegative": eod_trajectory.groupby("PositionID")["FastSlope"].apply(lambda x: (x < 0).mean() > 0.5),
    "FinishedPositive": eod_trajectory.sort_values("CandleOpen").groupby("PositionID")["FastSlope"].last() > 0,
})
summary.sum()

In [ ]:
finished_positive_ids = finished_positive[finished_positive].index
eod_losers_realized = positions[positions["ID"].isin(eod_positions)][["ID", "RealizedGain"]].rename(columns={"ID": "PositionID"})

eod_losers_realized["SlopeFinishedPositive"] = eod_losers_realized["PositionID"].isin(finished_positive_ids)
eod_losers_realized.groupby("SlopeFinishedPositive")["RealizedGain"].describe()

In [ ]:
winners_after_1pm = outcomes[outcomes["IsWin"] == 1]["PositionID"]

winner_trajectory = trade_signals[
    trade_signals["PositionID"].isin(winners_after_1pm)
][["PositionID", "CandleOpen", "FastSlope"]].sort_values(["PositionID", "CandleOpen"])

winner_summary = pd.DataFrame({
    "AllNegative": winner_trajectory.groupby("PositionID")["FastSlope"].apply(lambda x: (x < 0).all()),
    "MostlyNegative": winner_trajectory.groupby("PositionID")["FastSlope"].apply(lambda x: (x < 0).mean() > 0.5),
    "FinishedPositive": winner_trajectory.sort_values("CandleOpen").groupby("PositionID")["FastSlope"].last() > 0,
})
winner_summary.sum()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde

WIN_COLOR = "#0ca30c"
LOSS_COLOR = "#d03b3b"
GRID_COLOR = "#e1e0d9"
MUTED_INK = "#898781"

def plot_kde_by_outcome(ax, df, col):
    win = df.loc[df["IsWin"] == 1, col].dropna()
    loss = df.loc[df["IsWin"] == 0, col].dropna()

    lo, hi = df[col].quantile([0.01, 0.99])
    x = np.linspace(lo, hi, 300)

    for series, color, label in [(loss, LOSS_COLOR, "Loss"), (win, WIN_COLOR, "Win")]:
        kde = gaussian_kde(series)
        y = kde(x)
        ax.plot(x, y, color=color, linewidth=2, label=f"{label} (n={len(series)})")
        ax.fill_between(x, y, color=color, alpha=0.15)

    ax.axvline(win.median(), color=WIN_COLOR, linewidth=1, linestyle="--", alpha=0.6)
    ax.axvline(loss.median(), color=LOSS_COLOR, linewidth=1, linestyle="--", alpha=0.6)

    ax.set_title(col, fontsize=11, color="#0b0b0b")
    ax.set_ylabel("Density", fontsize=9, color=MUTED_INK)
    ax.tick_params(colors=MUTED_INK, labelsize=8)
    ax.grid(True, color=GRID_COLOR, linewidth=0.8)
    ax.set_axisbelow(True)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_color(GRID_COLOR)
    ax.legend(fontsize=8, frameon=False)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

plot_kde_by_outcome(axes[0, 0], buy_trade_signals, "FastDelta")
plot_kde_by_outcome(axes[0, 1], buy_trade_signals, "MovingAverageDelta")
plot_kde_by_outcome(axes[1, 0], buy_trade_signals, "VWAPDelta")

# OpenRangePos is categorical (-1/0/1), so win rate by bucket rather than a KDE
ax = axes[1, 1]
grp = buy_trade_signals.groupby("OpenRangePos")["IsWin"].agg(["mean", "count"]).reindex([-1, 0, 1])
overall = buy_trade_signals["IsWin"].mean()

bars = ax.bar(grp.index.astype(str), grp["mean"], color="#2a78d6", width=0.6)
ax.axhline(overall, color=MUTED_INK, linewidth=1, linestyle="--", alpha=0.7)
ax.text(1.35, overall, f"overall {overall:.0%}", fontsize=8, color=MUTED_INK, va="center")

for bar, (rate, n) in zip(bars, grp[["mean", "count"]].values):
    ax.text(bar.get_x() + bar.get_width() / 2, rate + 0.01, f"{rate:.0%}\n(n={int(n)})",
            ha="center", va="bottom", fontsize=8, color="#0b0b0b")

ax.set_title("OpenRangePos win rate", fontsize=11, color="#0b0b0b")
ax.set_ylabel("Win rate", fontsize=9, color=MUTED_INK)
ax.set_xticklabels(["Below (-1)", "Inside (0)", "Above (1)"])
ax.set_ylim(0, max(grp["mean"].max(), overall) + 0.15)
ax.tick_params(colors=MUTED_INK, labelsize=8)
ax.grid(True, axis="y", color=GRID_COLOR, linewidth=0.8)
ax.set_axisbelow(True)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["left", "bottom"]:
    ax.spines[spine].set_color(GRID_COLOR)

fig.tight_layout()
plt.show()

In [ ]:
import numpy as np
from scipy.stats import norm

def two_proportion_ztest(x1, n1, x2, n2):
    p1, p2 = x1 / n1, x2 / n2
    p_pool = (x1 + x2) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    z = (p1 - p2) / se
    p = 2 * (1 - norm.cdf(abs(z)))  # two-tailed
    return z, p

table = buy_trade_signals.groupby("OpenRangePos")["IsWin"].agg(["sum", "count"])
inside_wins, inside_n = table.loc[0, "sum"], table.loc[0, "count"]
edges_wins = table.loc[-1, "sum"] + table.loc[1, "sum"]
edges_n = table.loc[-1, "count"] + table.loc[1, "count"]

z, p = two_proportion_ztest(inside_wins, inside_n, edges_wins, edges_n)
print(f"z={z:.3f}, p={p:.3f}")

In [ ]:
buy_trade_signals.groupby("OpenRangePos")["RealizedGain"].agg(["mean", "median", "count"])

buy_trade_signals.groupby("OpenRangePos")["RealizedGain"].agg(["mean", "median", "std", "min", "max", "count"])